# x2b_yield-sojamilho — Rendimento Potencial Agroecológico GAEZ v4 (kg matéria seca/ha)

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** rendimento potencial de soja e milho (GAEZ), kg de matéria seca/ha, municipal, estático*  
**Conceito e Mecanismo Econômico:** Produtividade biofísica máxima estimada pelo modelo GAEZ (FAO/IIASA), baseada em clima de referência (1961-1990) e solo de 2010. Variável estática e exógena.  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h1__suitability_yield__gaez_v4_6190__candidatos.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Explorar variantes yx, por produto e alto/baixo insumo, em sequeiro; clima de referência 1961–1990.
> - Base de terras de 2010: validar sua adequação como capacidade predeterminada para 2001–2024.
> - Compatibilizar a base de umidade antes de comparar com PAM ou combinar com preços comerciais.

### Salvaguarda Territorial (141 Municípios de Mato Grosso):
> **Regra Obrigatória do Projeto:** O estado de Mato Grosso é considerado com exatamente **141 municípios**.  
> O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`), recentemente desmembrado de Nova Ubiratã (`5106240`) e Sorriso (`5107925`), tem seus dados **aglutinados no município original** (`5106240 - Nova Ubiratã`), preservando a consistência do painel longitudinal (Área Mínima Comparável - AMC).

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`codigo_ibge`**: Código do município (141 municípios).
- **`municipio`**: Nome do município.
- **`variavel_raster`**: Código da camada GAEZ (ex: yxHr0_soy = yield alto insumo soja).
- **`valor_medio`**: Produtividade média em kg de matéria seca por hectare (DW kg/ha).
- **`nivel_insumo`**: Nível de insumo (alto ou baixo).
- **`regime`**: Regime de sequeiro (r0).


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h1__suitability_yield__gaez_v4_6190__candidatos.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x2b_yield-sojamilho`:
- **Atenção:** Explorar variantes yx, por produto e alto/baixo insumo, em sequeiro; clima de referência 1961–1990.
- **Atenção:** Base de terras de 2010: validar sua adequação como capacidade predeterminada para 2001–2024.
- **Atenção:** Compatibilizar a base de umidade antes de comparar com PAM ou combinar com preços comerciais.
- **Salvaguarda Territorial:** Harmonização estrita para 141 municípios de Mato Grosso. Se houver dados de Boa Esperança do Norte (`5101837`), são aglutinados em Nova Ubiratã (`5106240`).


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
df.columns = df.columns.str.strip().str.lower().str.replace('\ufeff', '', regex=False)
df['codigo_ibge'] = df['codigo_ibge'].astype(str).str.zfill(7)
df['valor_medio'] = pd.to_numeric(df['valor_medio'], errors='coerce')

# HARMONIZAÇÃO TERRITORIAL (141 MUNICÍPIOS):
df['codigo_ibge'] = df['codigo_ibge'].replace({'5101837': '5106240'})

# REGRA DO TXT:
# 'Explorar variantes yx, por produto e alto/baixo insumo, em sequeiro; clima de referência 1961–1990.'
# 'Compatibilizar a base de umidade antes de comparar com PAM ou combinar com preços comerciais.'
df_yield = df[df['variavel_raster'].str.startswith('yx')].copy()
df_yield = df_yield.groupby(['codigo_ibge', 'variavel_raster'], as_index=False)['valor_medio'].mean()
print(f"Municípios únicos: {df_yield['codigo_ibge'].nunique()} (141 municípios)")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print("Descritiva por camada GAEZ (kg DW/ha):")
print(df_yield.groupby('variavel_raster')['valor_medio'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_yield, x='variavel_raster', y='valor_medio', palette='viridis')
plt.title("Rendimento Potencial GAEZ (kg matéria seca/ha) em MT")
plt.xticks(rotation=45)
plt.ylabel("kg DW / ha")
plt.tight_layout()
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
df_piv = df_yield.pivot(index='codigo_ibge', columns='variavel_raster', values='valor_medio')
if 'yxHr0_soy' in df_piv.columns and 'yxHr0_mze' in df_piv.columns:
    sns.scatterplot(data=df_piv, x='yxHr0_soy', y='yxHr0_mze', color='forestgreen')
    plt.title("Rendimento Potencial GAEZ Alto Insumo: Soja vs Milho")
    plt.xlabel("Soja (kg DW/ha)")
    plt.ylabel("Milho (kg DW/ha)")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

## Fase 6: Feature Engineering — Produtividade Potencial GAEZ v4 e Yield Gap

### 1. Step 1: Correção Agronômica de Umidade Comercial (Dry Weight -> 13% Umidade)
- **O Modelo GAEZ v4:** O modelo biofísico FAO/IIASA calcula a produtividade potencial em **peso seco puro** (Dry Weight - DW), ou seja, como se o grão não contivesse água alguma.
- **O Mercado Comercial de Soja:** A soja e o milho são colhidos, padronizados e comercializados com teor médio de **13% de umidade** (norma oficial do MAPA).
- **Fórmula de Conversão:**
  $$Yield_{comercial} = \frac{Yield_{GAEZ\_DW}}{0{,}87}$$
  Isso alinha perfeitamente a escala física do GAEZ com a escala dos dados do IBGE e dos preços comerciais.

### 2. Step 2: Cálculo da Brecha de Produtividade (Yield Gap)
- **Brecha de Produtividade:** $YieldGap_{i, t} = \frac{\text{Rendimento Observado}_{i, t-1}}{Yield_{comercial, i}}$.
- Se o Yield Gap for próximo de 1, o município atingiu a fronteira tecnológica; se for baixo (ex: 0,5), há espaço para dobrar a colheita na mesma área sem abrir novas terras (poupança de terra / intensificação).


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Pivotar o dataset GAEZ para ter os potenciais em colunas
df_piv = df.pivot_table(index='codigo_ibge', columns='variavel_raster', values='valor_medio').reset_index()

# 2. Correção de Umidade Comercial: Matéria Seca (DW) / 0.87 = Base Comercial 13% Umidade
# gaez_yield_soja_comercial_kg_ha: rendimento potencial agroclimático de soja em base comercial (kg/ha)
# gaez_yield_milho_comercial_kg_ha: rendimento potencial agroclimático de milho em base comercial (kg/ha)
if 'yxHr0_soy' in df_piv.columns:
    df_piv['gaez_yield_soja_comercial_kg_ha'] = df_piv['yxHr0_soy'] / 0.87
if 'yxHr0_miz' in df_piv.columns:
    df_piv['gaez_yield_milho_comercial_kg_ha'] = df_piv['yxHr0_miz'] / 0.87

# 3. Código-esqueleto para Yield Gap no Painel:
# yield_gap_soja = rendimento_observado_lag1 / gaez_yield_soja_comercial_kg_ha
df_piv['exemplo_yield_gap_soja'] = np.nan

print(f"Municípios com rendimento potencial GAEZ ajustado: {len(df_piv)}")
print(df_piv[['codigo_ibge', 'gaez_yield_soja_comercial_kg_ha', 'gaez_yield_milho_comercial_kg_ha']].head())